# SFDC ↔ Airtable Job Reconciliation

Reconstructs the AT-vs-SFDC reconciliation table (the Google-Sheet exercise) directly from
the **Airtable API** + a **Salesforce report CSV**, and emits a row-level list of the jobs
whose attributes disagree between the two systems.

**Inputs**
1. A Salesforce report CSV in the format you paste in (one row per *Site Service*).
   Drop new versions into `data/` and point `SFDC_CSV_PATH` at the latest.
2. The Airtable **Job Tracking** table (base `Command Context Sync`), pulled live via API.

**Outputs**
1. `reconciliation_summary` — the AT / SFDC side-by-side metric table (Customers, Locations,
   Rows/SS, Jobs, Sqft) for every Status × Filter × Customer × Type combination in the sheet,
   plus the *Adjusted-for-Prime* bridge and the **Delta** row.
2. `job_diffs.md` — a bulleted list of every job whose fields differ, in the form:

   ```
   * <site>
       * <attribute>: X on SFDC, Y on AT
   ```

**Join key:** SFDC `Job Name` (e.g. `JOB-02043`) ⇄ Airtable **`new_sfdc_job_sync`** — the direct
SFDC→Airtable sync field, rather than the derived `18char Job ID Rollup`.

> The numbers will drift slightly from any given screenshot because the CSV and the Airtable
> pull are taken at different moments — that drift *is* the thing we are measuring.


## 1. Configuration

Secrets are loaded from a local **`.env`** file (kept out of git) and then the environment.
Create `.env` next to this notebook:

```dotenv
AIRTABLE_API_TOKEN=patXXXXXXXX...
AIRTABLE_ENTERPRISE_ACCOUNT_ID=entXXXXXXXX   # optional, only for audit-log attribution
```

All base/table/field IDs below were resolved from the live schema, so nothing here relies on
display names that someone might rename in the Airtable UI.

In [ ]:
import os, re, json, time, glob, datetime as dt
from collections import defaultdict
import requests
import pandas as pd

# ---- Load .env (no hard dependency on python-dotenv) ---------------------------------------
def load_dotenv(path=".env"):
    try:
        from dotenv import load_dotenv as _ld   # use python-dotenv if present
        _ld(path)
        return
    except Exception:
        pass
    if os.path.exists(path):
        for line in open(path):
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

load_dotenv()

# ---- Airtable identifiers (resolved from the live schema) ----------------------------------
AIRTABLE_API_TOKEN = (
    os.environ.get("AIRTABLE_API_TOKEN")
    or os.environ.get("AIRTABLE_API_KEY")
    or os.environ.get("AIRTABLE_TOKEN")
)
BASE_ID  = "app7jMwevErzRNk7G"      # Command Context Sync
TABLE_ID = "tblv8G1wvbMyzJpJd"      # Job Tracking

# Airtable field IDs we need (id -> friendly name). Using IDs survives field renames.
AT_FIELDS = {
    "job_id":        "fld8yRsobul4XnOuN",  # 18char Job ID Rollup (from new_sfdc_job_sync) -- ref only
    "job_sync":      "fldzaN7ZUNFuQCrdI",  # new_sfdc_job_sync (= SFDC Job Name, e.g. JOB-01283) <-- JOIN KEY
    "account":       "fldXxKR4zpyjpgEba",  # new_sfdc_account_name
    "address":       "fldtw7MdvgzEDCgFd",  # new_sfdc_concat_full_address
    "sqft":          "fld5s9VX2hp8CLr27",  # Actual Square Footage Rollup (from new_sfdc_ol_sync)
    "total_sqft":    "fld3rjmwpIisZncjZ",  # Total Sq. Ft.
    "job_status":    "fldVkbhy7xLFLWGNN",  # Job Status
    "job_type":      "fldEaFaSjZfq1BEWe",  # Job Type
    "network_live":  "fld6ehsRla0PEYvUQ",  # Network Live? (Jobs)  (checkbox)
    "target_golive": "fldujSlIehKhZXvbN",  # Target Go-Live Date (Meter)
    "actual_golive": "fldDvLA5uaH5gWa2J",  # Actual Go-Live Date
    "is_prime":      "fld9hprabcxAZ9IWX",  # is_prime  (1 = Prime Group Holdings)
    "connect_cust":  "fldwcdAQtzdGu8TiL",  # Connect customer?
    "job_name":      "fldWZE2JbBTALWlmg",  # Job ID  (JOB-xxxx label, for readability)
}

# ---- SFDC report CSV -----------------------------------------------------------------------
SFDC_CSV_PATH = "data/sfdc_report_2026-06-22.csv"   # <-- point at the latest export
SFDC_CSV_ENCODING = "latin-1"                       # the SFDC export is not clean utf-8

# SFDC column names (left) -> canonical name (right)
SFDC_COLS = {
    "18char Job ID":                    "job_id",
    "Job: Account Name":                "account",
    "OL: Full Address":                 "address",
    "Actual Square Footage":            "sqft",
    "Job: Job Status":                  "job_status",
    "Job: Job Type":                    "job_type",
    "Job: Network Live?":               "network_live",
    "Job: Target Go-Live Date (Meter)": "target_golive",
    "Actual Go-Live Date":              "actual_golive",
    "Site Service: Site Service Name":  "site_service",
    "Job Name":                         "job_name",
    "ProvSite: Opportunity Name":       "opportunity",
    "Cellular + Connect Only":          "cellular_connect",
}

# ---- Business rules (edit here if the report definition changes) ---------------------------
# "Focused" filter set, minus the dimensions broken out as their own columns (Status / Type).
EXCLUDED_ACCOUNTS   = {"Amrize", "Meter", "Lineage Logistics"}
EXCLUDED_STATUSES   = {"Complete", "Canceled", "Cancelled"}
# Job Types removed by the full "Focused" definition:
NON_FOCUSED_TYPES   = {"Pre-install", "Connect-only", "Cellular-only", "NFR"}
# "Adjusted" type column == everything except Pre-install:
PREINSTALL_TYPES    = {"Pre-install"}
PRIME_ACCOUNTS      = {"Prime Group Holdings LLC"}  # the "Prime" carved out of the bridge

assert AIRTABLE_API_TOKEN, "Set AIRTABLE_API_TOKEN in the environment before running."
print("Config loaded. CSV:", SFDC_CSV_PATH)

## 2. Pull the Airtable Job Tracking table

Straight REST (`GET /v0/{base}/{table}`) with cursor pagination, requesting only the fields we
need. ~4,900 records, so this is a handful of pages.

In [ ]:
def fetch_airtable_records(base_id, table_id, field_ids, token, page_size=100):
    """Yield every record from a table, requesting only `field_ids`."""
    url = f"https://api.airtable.com/v0/{base_id}/{table_id}"
    headers = {"Authorization": f"Bearer {token}"}
    params = [("pageSize", page_size), ("returnFieldsByFieldId", "true")]
    params += [("fields[]", fid) for fid in field_ids]
    offset = None
    while True:
        p = list(params) + ([("offset", offset)] if offset else [])
        for attempt in range(5):
            r = requests.get(url, headers=headers, params=p, timeout=60)
            if r.status_code == 429:          # rate limited
                time.sleep(2 ** attempt)
                continue
            r.raise_for_status()
            break
        data = r.json()
        for rec in data["records"]:
            yield rec
        offset = data.get("offset")
        if not offset:
            break

def _scalar(v):
    """Flatten Airtable cell values to plain scalars."""
    if isinstance(v, dict):                      # singleSelect / collaborator
        return v.get("name", v.get("id"))
    if isinstance(v, list):                      # multiSelect / rollup / link
        if not v:
            return None
        return ", ".join(_scalar(x) for x in v if x is not None)
    return v

raw = list(fetch_airtable_records(BASE_ID, TABLE_ID, list(AT_FIELDS.values()), AIRTABLE_API_TOKEN))
print(f"Pulled {len(raw)} Airtable records")

rows = []
inv = {fid: name for name, fid in AT_FIELDS.items()}
for rec in raw:
    f = rec.get("fields", {})
    row = {"at_record_id": rec["id"]}
    for fid, name in inv.items():
        row[name] = _scalar(f.get(fid))
    rows.append(row)
at = pd.DataFrame(rows)
at.head(3)

## 3. Load & normalise both sides

We coerce both frames to a shared canonical schema so every downstream comparison is
apples-to-apples. The join key is normalised (trimmed, upper-cased).

In [ ]:
def norm_id(x):
    return str(x).strip().upper() if pd.notna(x) and str(x).strip() else None

def norm_text(x):
    if pd.isna(x):
        return ""
    return re.sub(r"\s+", " ", str(x)).strip()

def to_float(x):
    if pd.isna(x) or str(x).strip() == "":
        return 0.0
    try:
        return float(str(x).replace(",", ""))
    except ValueError:
        return 0.0

def to_bool_live(x):
    # SFDC exports "0"/"1"; Airtable checkbox -> True/None
    if isinstance(x, bool):
        return x
    return str(x).strip() in {"1", "True", "true", "Yes"}

# ---- SFDC ----------------------------------------------------------------------------------
# sfdc_raw keeps the ORIGINAL CSV columns/order (used to emit the same-shape diff frame later).
sfdc_raw = pd.read_csv(SFDC_CSV_PATH, dtype=str, encoding=SFDC_CSV_ENCODING).fillna("")
sfdc = sfdc_raw.rename(columns=SFDC_COLS)[list(SFDC_COLS.values())].copy()
sfdc["job_id_key"]    = sfdc["job_id"].map(norm_id)
sfdc["job_key"]       = sfdc["job_name"].map(norm_id)   # JOB-xxxx  <-- JOIN KEY (matches new_sfdc_job_sync)
sfdc["network_live"]  = sfdc["network_live"].map(to_bool_live)
sfdc["sqft_num"]      = sfdc["sqft"].map(to_float)
sfdc["cellular_connect_num"] = sfdc["cellular_connect"].map(to_float)
sfdc["source"] = "SFDC"

# ---- Airtable ------------------------------------------------------------------------------
at["job_id_key"]   = at["job_id"].map(norm_id)
at["job_key"]      = at["job_sync"].map(norm_id)        # new_sfdc_job_sync (JOB-xxxx)  <-- JOIN KEY
at["network_live"] = at["network_live"].map(to_bool_live)
at["sqft_num"]     = at["sqft"].map(to_float)
at["is_prime_num"] = at["is_prime"].map(to_float)
at["site_service"] = ""          # AT grain is the site-service row already
at["cellular_connect_num"] = at["connect_cust"].map(lambda v: 1.0 if norm_text(v) else 0.0)
at["source"] = "AT"

print("SFDC rows:", len(sfdc), "| Airtable rows:", len(at))
print("Join key = new_sfdc_job_sync (JOB-xxxx).")
print("  SFDC rows with a job_key:", sfdc["job_key"].notna().sum())
print("  AT  rows with a job_key :", at["job_key"].notna().sum(),
      f"({at['job_key'].isna().sum()} AT rows have no new_sfdc_job_sync)")

## 4. Filter predicates

Each dimension in the reconciliation sheet is a predicate. They compose, so any
Status × Filter × Customer × Type combination from the screenshots can be requested by name.

| Dimension | Values | Rule |
|---|---|---|
| **Status** | `Live` / `Pre-Live` | `Network Live?` == True / False |
| **Filters** | `None` / `Focused` | Focused ⇒ drop excluded accounts, excluded statuses, cellular+connect, non-focused job types |
| **Customer** | `All` / a specific account | exact account-name match |
| **Type** | `All` / `Adjusted` / `Pre-Install` | Adjusted ⇒ Job Type ≠ Pre-install; Pre-Install ⇒ Job Type == Pre-install |


In [ ]:
def predicate(df, *, status=None, filters="None", customer="All", type_="All",
              drop_prime=False):
    m = pd.Series(True, index=df.index)

    if status == "Live":
        m &= df["network_live"] == True
    elif status == "Pre-Live":
        m &= df["network_live"] == False

    if filters == "Focused":
        m &= ~df["account"].isin(EXCLUDED_ACCOUNTS)
        m &= ~df["job_status"].isin(EXCLUDED_STATUSES)
        m &= df["cellular_connect_num"] != 1
        m &= ~df["job_type"].isin(NON_FOCUSED_TYPES)

    if customer != "All":
        m &= df["account"] == customer

    if type_ == "Adjusted":
        m &= ~df["job_type"].isin(PREINSTALL_TYPES)
    elif type_ == "Pre-Install":
        m &= df["job_type"].isin(PREINSTALL_TYPES)

    if drop_prime:
        m &= ~df["account"].isin(PRIME_ACCOUNTS)

    return df[m]

def metrics(df):
    """The five reconciliation measures for a filtered frame."""
    return {
        "Customers": df["account"].replace("", pd.NA).nunique(),
        "Locations": df["address"].map(norm_text).replace("", pd.NA).nunique(),
        "Rows":      len(df),
        "Jobs":      df["job_id_key"].nunique(),
        "Sqft":      int(df["sqft_num"].sum()),
    }

## 5. Reconstruct the reconciliation summary

This reproduces the row layout of the AIRTABLE / SFDC blocks in the sheet. Edit `COMBINATIONS`
to add or drop rows.

In [ ]:
COMBINATIONS = [
    # label,                              kwargs
    ("Live  | None     | All",            dict(status="Live",     filters="None")),
    ("PreLv | None     | All",            dict(status="Pre-Live", filters="None")),
    ("Live  | Focused  | All",            dict(status="Live",     filters="Focused")),
    ("PreLv | Focused  | All",            dict(status="Pre-Live", filters="Focused")),
    ("Live  | Focused  | Adjusted",       dict(status="Live",     filters="Focused", type_="Adjusted")),
    ("PreLv | Focused  | Adjusted",       dict(status="Pre-Live", filters="Focused", type_="Adjusted")),
    ("Live  | Focused  | Pre-Install",    dict(status="Live",     filters="Focused", type_="Pre-Install")),
    ("PreLv | Focused  | Pre-Install",    dict(status="Pre-Live", filters="Focused", type_="Pre-Install")),
    ("Live  | All      | Pre-Install",    dict(status="Live",     filters="None",    type_="Pre-Install")),
    ("PreLv | All      | Pre-Install",    dict(status="Pre-Live", filters="None",    type_="Pre-Install")),
]

def build_summary(combinations):
    out = []
    for label, kw in combinations:
        a = metrics(predicate(at, **kw))
        s = metrics(predicate(sfdc, **kw))
        row = {"Combination": label}
        for k in ["Customers", "Locations", "Rows", "Jobs", "Sqft"]:
            row[f"AT.{k}"]    = a[k]
            row[f"SFDC.{k}"]  = s[k]
            row[f"Δ.{k}"]     = a[k] - s[k]
        out.append(row)
    return pd.DataFrame(out)

reconciliation_summary = build_summary(COMBINATIONS)
pd.set_option("display.max_columns", None, "display.width", 200)
reconciliation_summary

### 5b. The "Adjusted for Prime" bridge + Delta

The orange *Comparison* block: Pre-Live, Focused, Adjusted, **Prime Group Holdings removed**,
so the two systems can be lined up without Prime's large-but-volatile footprint skewing it.

In [ ]:
bridge_kw = dict(status="Pre-Live", filters="Focused", type_="Adjusted", drop_prime=True)
at_bridge   = metrics(predicate(at,   **bridge_kw))
sfdc_bridge = metrics(predicate(sfdc, **bridge_kw))

bridge = pd.DataFrame(
    [at_bridge, sfdc_bridge, {k: at_bridge[k] - sfdc_bridge[k] for k in at_bridge}],
    index=["AT Adj. for Prime", "SFDC Adj. for Prime", "Delta (AT - SFDC)"],
)
bridge

## 6. Row-level diff — `diffs_df` (same shape as the input CSV) + attribute detail

Join key is **`new_sfdc_job_sync`** (the SFDC Job Name, `JOB-xxxx`) ⇄ the CSV's `Job Name` — the
*direct* sync field, not the derived 18-char rollup. SFDC stays at its native **site-service** grain;
Airtable is collapsed to one row per job (first non-empty value per attribute). Each site service is
compared, attribute by attribute, against its Airtable job.

Outputs:
- **`diffs_df`** — the differing rows in the **exact shape of the input CSV** (identical columns),
  written to `runs/sfdc_diffs.csv`. Re-uploadable / shareable as-is.
- **`diff_detail`** — long format: one row per (site service × differing attribute), → `runs/sfdc_diff_detail.csv`.
- **`job_diffs.md`** — the bulleted `* site / * attribute: X on SFDC, Y on AT` list.

In [ ]:
# Attributes to compare, and how to render each value.
COMPARE_ATTRS = {
    "account":       ("Account",        norm_text),
    "address":       ("Address",        norm_text),
    "sqft":          ("Sq Ft",          lambda v: f"{to_float(v):,.0f}"),
    "job_status":    ("Job Status",     norm_text),
    "job_type":      ("Job Type",       norm_text),
    "target_golive": ("Target Go-Live", norm_text),
    "actual_golive": ("Actual Go-Live", norm_text),
    "network_live":  ("Network Live?",  lambda v: str(bool(v))),
}

def first_nonempty(s):
    for v in s:
        if pd.notna(v) and str(v).strip() not in ("", "nan"):
            return v
    return s.iloc[0] if len(s) else None

# Airtable collapsed to one row per job, keyed on new_sfdc_job_sync (job_key).
at_cols   = [c for c in COMPARE_ATTRS if c in at.columns] + ["job_id_key", "job_name"]
at_cols   = [c for c in at_cols if c in at.columns]
at_by_job = at.dropna(subset=["job_key"]).groupby("job_key")[at_cols].agg(first_nonempty)

sfdc_keys = set(sfdc["job_key"].dropna())
at_keys   = set(at_by_job.index)
shared    = sorted(sfdc_keys & at_keys)
only_sfdc = sorted(sfdc_keys - at_keys)
only_at   = sorted(at_keys - sfdc_keys)
print(f"Join on new_sfdc_job_sync — shared jobs: {len(shared)} | only SFDC: {len(only_sfdc)} | only AT: {len(only_at)}")

def render_val(attr, v):
    fmt = COMPARE_ATTRS[attr][1]
    try:
        return fmt(v)
    except Exception:
        return norm_text(v)

# Per-site-service comparison (SFDC keeps its native grain so the output matches the CSV shape).
row_differs = pd.Series(False, index=sfdc.index)
diff_detail_rows = []
diffs = []          # (site_label, job_key, [(attr_label, sfdc_val, at_val), ...]) -- for sticky tracking
for idx, s in sfdc.iterrows():
    jk = s["job_key"]
    if jk not in at_keys:
        continue                                   # missing-in-AT handled via only_sfdc, not as an attr diff
    a = at_by_job.loc[jk]
    rowdiffs = []
    for attr, (label, _) in COMPARE_ATTRS.items():
        if attr not in sfdc.columns or attr not in at_by_job.columns:
            continue
        if attr == "sqft":
            if abs(to_float(s.get("sqft")) - to_float(a.get("sqft"))) > 0.5:
                rowdiffs.append((label, render_val(attr, s.get("sqft")), render_val(attr, a.get("sqft"))))
            continue
        sv, av = render_val(attr, s.get(attr)), render_val(attr, a.get(attr))
        if sv != av:
            rowdiffs.append((label, sv, av))
    if rowdiffs:
        row_differs.loc[idx] = True
        site = (str(s.get("site_service") or "").strip() or str(s.get("job_name") or "").strip() or str(jk))
        addr = norm_text(s.get("address"))
        diffs.append((f"{site} — {addr}" if addr else site, jk, rowdiffs))
        for label, sv, av in rowdiffs:
            diff_detail_rows.append({"site_service": s.get("site_service"), "job_name": s.get("job_name"),
                                     "job_key": jk, "attribute": label, "sfdc_value": sv, "at_value": av})

# (1) SAME SHAPE AS INPUT CSV: original columns, only the differing site-service rows.
diffs_df = sfdc_raw.loc[row_differs[row_differs].index].copy()
# (2) long-format attribute detail
diff_detail = pd.DataFrame(diff_detail_rows,
                           columns=["site_service","job_name","job_key","attribute","sfdc_value","at_value"])

import os
os.makedirs("runs", exist_ok=True)
diffs_df.to_csv("runs/sfdc_diffs.csv", index=False)
diff_detail.to_csv("runs/sfdc_diff_detail.csv", index=False)
print(f"Differing site services: {len(diffs_df)} of {len(sfdc)} "
      f"(across {len({jk for _, jk, _ in diffs})} jobs)")
print(f"runs/sfdc_diffs.csv  -> same shape as input CSV: {diffs_df.shape[0]} rows x {diffs_df.shape[1]} cols")
diffs_df.head(10)

In [ ]:
# Build the markdown bullet list in the requested shape.
lines = []
lines.append(f"# SFDC ↔ AT diffs  ({dt.date.today().isoformat()})  — join: new_sfdc_job_sync\n")
lines.append(f"- Shared jobs compared: **{len(shared)}**")
lines.append(f"- Differing site services: **{len(diffs_df)}** (across {len({jk for _, jk, _ in diffs})} jobs)")
lines.append(f"- Jobs only in SFDC: **{len(only_sfdc)}**  |  only in Airtable: **{len(only_at)}**\n")

lines.append("## Attribute differences\n")
for site_label, jk, rowdiffs in diffs:
    lines.append(f"* {site_label}  `({jk})`")
    for label, sv, av in rowdiffs:
        lines.append(f"   * {label}: {sv} on SFDC, {av} on AT")

if only_sfdc:
    sfdc_by_job = (sfdc.dropna(subset=["job_key"])
                       .groupby("job_key")[["site_service", "job_name", "address"]].agg(first_nonempty))
    lines.append("\n## Jobs present in SFDC but missing from Airtable\n")
    for jk in only_sfdc:
        s = sfdc_by_job.loc[jk]
        site = str(s.get("site_service") or s.get("job_name") or "").strip()
        lines.append(f"* {site} `({jk})` — {norm_text(s.get('address'))}")

if only_at:
    lines.append("\n## Jobs present in Airtable but missing from SFDC\n")
    for jk in only_at:
        a = at_by_job.loc[jk]
        lines.append(f"* {jk} — {norm_text(a.get('address'))}")

md_text = "\n".join(lines)
with open("job_diffs.md", "w") as fh:
    fh.write(md_text)
print("Wrote job_diffs.md")
print("\n".join(md_text.splitlines()[:40]))

## 7. Persist the run + find *sticky* diffs

This is the over-time layer, kept deliberately cheap:

- **Trend line** — append this run's Adjusted-for-Prime metrics + diff counts to
  `runs/run_history.csv` (one row per run). That's the "is the gap shrinking?" series, for free.
- **Diff snapshot** — write this run's differing job IDs to `runs/diffs_<timestamp>.json`.
- **Sticky diffs** — jobs that differ in *this* run **and** the previous one. Transient sync lag
  clears itself between runs; a sticky diff is genuine drift, and that's the only thing worth
  spending an audit-log call on (Section 8).

In [ ]:
RUNS_DIR = "runs"
os.makedirs(RUNS_DIR, exist_ok=True)
run_ts = dt.datetime.now().strftime("%Y%m%dT%H%M%S")
diff_job_ids = sorted({jid for _, jid, _ in diffs})

# Most recent prior snapshot (read BEFORE writing this run's, so it's the previous run).
prior = sorted(glob.glob(os.path.join(RUNS_DIR, "diffs_*.json")))
prior_ids = set()
if prior:
    prior_ids = set(json.load(open(prior[-1])).get("diff_job_ids", []))

# 1) trend line
record = {"run_ts": run_ts, "sfdc_csv": SFDC_CSV_PATH}
for who, m in [("AT", at_bridge), ("SFDC", sfdc_bridge)]:
    for k, v in m.items():
        record[f"{who}.{k}"] = v
record.update(n_diff_jobs=len(diff_job_ids), n_only_sfdc=len(only_sfdc), n_only_at=len(only_at))
hist_path = os.path.join(RUNS_DIR, "run_history.csv")
pd.DataFrame([record]).to_csv(hist_path, mode="a", header=not os.path.exists(hist_path), index=False)

# 2) this run's diff snapshot
json.dump({"run_ts": run_ts, "sfdc_csv": SFDC_CSV_PATH, "diff_job_ids": diff_job_ids,
           "only_sfdc": only_sfdc, "only_at": only_at},
          open(os.path.join(RUNS_DIR, f"diffs_{run_ts}.json"), "w"), indent=1)

# 3) sticky = differs now AND differed in the previous run
sticky_job_ids = sorted(set(diff_job_ids) & prior_ids)
print(f"Run {run_ts}: {len(diff_job_ids)} diffing jobs | {len(sticky_job_ids)} sticky "
      f"(also differed in {os.path.basename(prior[-1]) if prior else 'n/a — first run'})")
pd.read_csv(hist_path)

## 8. Transition log — the `changeEvents` endpoint (the SSOT change tracker)

`GET /v0/meta/enterpriseAccounts/{enterpriseAccountId}/changeEvents` is the real-time, **cell-level**
change feed. It is the right source for "track every status / job / state change as it happens" —
unlike the audit log (a security trail), `changeEvents` carries field-level **before/after** values,
which is what a trustworthy changelog needs.

Two documented properties drive the design:

- **~14-day retention** — events age out, so this must be *polled and persisted*, not queried on
  demand. We append to a durable `runs/change_events.jsonl` and checkpoint the cursor.
- **Cursor pagination** — each poll resumes from the saved cursor, so it's incremental and dupe-free.

Drift attribution then becomes a *lookup against the log* ("which system moved this field, when, who"),
not a fresh API call per question.

> The enterprise `changeEvents` model docs are auth-gated, so the exact JSON nesting can't be verified
> from here. The poller below depends only on the documented cursor/retention behavior and **prints the
> first event's raw shape** so you can confirm the field paths in `normalize_change_event` on first run.

In [ ]:
ENTERPRISE_ACCOUNT_ID = os.environ.get("AIRTABLE_ENTERPRISE_ACCOUNT_ID")  # entXXXXXXXX (.env)
CHANGE_LOG   = os.path.join(RUNS_DIR, "change_events.jsonl")   # durable raw feed
CURSOR_PATH  = os.path.join(RUNS_DIR, "change_events.cursor")  # checkpoint for incremental polling

def _read_cursor():
    return open(CURSOR_PATH).read().strip() if os.path.exists(CURSOR_PATH) else None

def _write_cursor(c):
    if c is not None:
        open(CURSOR_PATH, "w").write(str(c))

def poll_change_events(account_id, token, *, cursor=None, limit=100, max_pages=500):
    """Incrementally pull the enterprise changeEvents feed forward from `cursor`.
    Returns (new_events, next_cursor). Handles both the cursor/mightHaveMore and the
    pagination.next response shapes. Persist regularly — events retain ~14 days."""
    url = f"https://api.airtable.com/v0/meta/enterpriseAccounts/{account_id}/changeEvents"
    headers = {"Authorization": f"Bearer {token}"}
    events, pages = [], 0
    while pages < max_pages:
        params = {"limit": limit}
        if cursor is not None:
            params["cursor"] = cursor
        r = requests.get(url, headers=headers, params=params, timeout=60)
        r.raise_for_status()
        data = r.json()
        batch = data.get("changeEvents") or data.get("events") or data.get("payloads") or []
        events.extend(batch)
        cursor = data.get("cursor", (data.get("pagination") or {}).get("next", cursor))
        pages += 1
        if not (data.get("mightHaveMore") or (data.get("pagination") or {}).get("next")):
            break
    return events, cursor

def normalize_change_event(ev):
    """Flatten one changeEvent into transition rows (one per changed cell).
    CONFIRM these paths against the first-event dump printed below — the model docs are gated."""
    ts    = ev.get("timestamp") or ev.get("createdTime")
    actor = (((ev.get("actor") or {}).get("user") or {}).get("email")
             or (ev.get("actor") or {}).get("type"))
    action = ev.get("action") or ev.get("type")
    rows = []
    for ch in (ev.get("changes") or ev.get("recordChanges") or [ev]):
        base_id = ch.get("baseId") or (ch.get("base") or {}).get("id")
        rid  = ch.get("recordId") or ch.get("id")
        cur  = ch.get("currentCellValuesByFieldId")  or ch.get("current")  or {}
        prev = ch.get("previousCellValuesByFieldId") or ch.get("previous") or {}
        for fid in (set(cur) | set(prev)):
            rows.append({"timestamp": ts, "actor": actor, "action": action,
                         "base_id": base_id, "record_id": rid, "field_id": fid,
                         "previous": prev.get(fid), "current": cur.get(fid)})
    return rows

if not ENTERPRISE_ACCOUNT_ID:
    print("Set AIRTABLE_ENTERPRISE_ACCOUNT_ID in .env to build the transition log.")
else:
    new_events, next_cursor = poll_change_events(
        ENTERPRISE_ACCOUNT_ID, AIRTABLE_API_TOKEN, cursor=_read_cursor())
    with open(CHANGE_LOG, "a") as fh:
        for ev in new_events:
            fh.write(json.dumps(ev) + "\n")
    _write_cursor(next_cursor)
    print(f"Polled {len(new_events)} new change events; cursor -> {next_cursor}")
    if new_events:
        print("\n--- first event raw shape (confirm normalize_change_event paths against this) ---")
        print(json.dumps(new_events[0], indent=2)[:1500])

### 8b. Attribute sticky diffs from the transition log

With the feed persisted, attributing a *sticky* diff is a lookup: pull the Job Tracking transitions
for the sticky records and read off which field moved, when, and by whom — so you can say whether
Airtable drifted after the last SFDC pull (or vice-versa) without another API call.

In [ ]:
def load_transition_log(path=CHANGE_LOG):
    if not os.path.exists(path):
        return pd.DataFrame(columns=["timestamp","actor","action","base_id","record_id","field_id","previous","current"])
    rows = []
    for line in open(path):
        line = line.strip()
        if line:
            rows.extend(normalize_change_event(json.loads(line)))
    df = pd.DataFrame(rows)
    if len(df) and "base_id" in df:
        df = df[df["base_id"].isin([BASE_ID, None])]
    return df

transitions = load_transition_log()
print(f"Transition log rows: {len(transitions)}")

if len(transitions) and sticky_job_ids:
    inv_at = {fid: name for name, fid in AT_FIELDS.items()}
    sticky_record_ids = set(at[at["job_key"].isin(sticky_job_ids)]["at_record_id"])
    sticky_tx = transitions[transitions["record_id"].isin(sticky_record_ids)].copy()
    sticky_tx["field"] = sticky_tx["field_id"].map(lambda f: inv_at.get(f, f))
    print(f"{len(sticky_tx)} logged transitions touch a sticky-diff record")
    display(sticky_tx.sort_values("timestamp", ascending=False)
                     [["timestamp","actor","field","previous","current","record_id"]].head(40))
else:
    print("No transition history yet for the current sticky diffs "
          "(poll changeEvents over a few days to accumulate one).")

## Do we need to scan this over time?

**No full time-scan needed — and the notebook now bakes in the lightweight version of it.**

The flow is: **point-in-time reconcile → diff → persist → attribute only what sticks.**

- **The reconciliation is a snapshot.** "These N jobs disagree today" is the actionable
  deliverable (Sections 5–6). You fix them, re-run, the list shrinks. You don't need history to *act*.
- **Most of today's delta is sync lag, not real divergence.** SFDC is exported manually; Airtable
  is live. Many diffs clear themselves on the next sync. The point of tracking over time is to
  *separate* transient lag from genuine drift — which is exactly what the **sticky** logic in
  Section 7 does (differs in two consecutive runs).
- **Two over-time needs, only one of which touches the audit log:**
  1. *Trend of the gap* — is it shrinking? → `runs/run_history.csv`, appended every run (Section 7).
     Cheap, no audit log.
  2. *Attribution* — who/what changed a field, and was it after the last SFDC pull? → the
     **`changeEvents` transition log** (Section 8), polled + persisted, then read **only for sticky
     diffs**, never re-scanned across the whole base.

So you do **not** need a continuous full-base scan. Run the notebook per CSV drop; the trend line
accrues for free; and the audit log is spent only on diffs that refuse to clear. That's "scan over
time" reduced to its useful core.
